# Threads Reader Bot → Cloudflare

Безопасное развёртывание по ячейкам из Google Colab.

**Важно:**
- cookies загружаются только во временную файловую систему текущей Colab-сессии;
- cookies, Telegram-токен и Cloudflare API Token не выводятся и не записываются в Git;
- cookies импортируются напрямую в D1, после чего локальные копии удаляются;
- Worker продолжает автоматически сохранять обновлённые cookies в D1 после успешного чтения Threads;
- завершите работу ячейкой «Очистка секретов» и удалите Colab runtime после установки.


## 1. Получение исходного кода

Пока Cloudflare-версия находится в рабочей ветке исходного репозитория. После переноса в `threadsbot-cloudflare` замените две переменные ниже на URL нового репозитория и `main`.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Bergaff/threadsbot.git"
REPO_BRANCH = "arena/01a0055c-threadsbot"
PROJECT = Path("/content/threadsbot-cloudflare")

if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run(["npm", "ci"], check=True)
print("✅ Исходники и зависимости готовы")


## 2. Cloudflare-доступ

Создайте API Token в Cloudflare с правами редактирования **Workers Scripts, D1, Queues и Browser Rendering**, а также скопируйте Account ID. Ввод скрыт через `getpass`; значения не сохраняются в ноутбуке.


In [ ]:
import getpass, os

os.environ["CLOUDFLARE_API_TOKEN"] = getpass.getpass("Cloudflare API Token: ").strip()
os.environ["CLOUDFLARE_ACCOUNT_ID"] = getpass.getpass("Cloudflare Account ID: ").strip()
assert os.environ["CLOUDFLARE_API_TOKEN"] and os.environ["CLOUDFLARE_ACCOUNT_ID"]
subprocess.run(["npx", "wrangler", "whoami"], check=True)
print("✅ Cloudflare подключён")


## 3. Создание D1 и Queue

Ячейка безопасна для повторного запуска: существующие `threadsbot` и `threadsbot-updates` используются повторно.


In [ ]:
import json, re, subprocess
from pathlib import Path

def run(*args, capture=False):
    return subprocess.run(list(args), cwd=PROJECT, check=True, text=True,
                          capture_output=capture)

# D1: находим существующую базу или создаём новую в Восточной Европе.
listed = run("npx", "wrangler", "d1", "list", "--json", capture=True)
databases = json.loads(listed.stdout)
db = next((x for x in databases if x.get("name") == "threadsbot"), None)
if db is None:
    created = run("npx", "wrangler", "d1", "create", "threadsbot", "--location", "eeur", capture=True)
    match = re.search(r'database_id\s*=\s*"([0-9a-f-]{36})"', created.stdout + created.stderr)
    if not match:
        raise RuntimeError("Не удалось определить database_id:\n" + created.stdout + created.stderr)
    database_id = match.group(1)
else:
    database_id = db.get("uuid") or db.get("id")

# Подставляем UUID без печати других настроек.
config = PROJECT / "wrangler.toml"
value = config.read_text("utf-8")
value, count = re.subn(r'database_id\s*=\s*"[^"]+"', f'database_id = "{database_id}"', value, count=1)
if count != 1:
    raise RuntimeError("Поле database_id не найдено в wrangler.toml")
config.write_text(value, "utf-8")

# Queue: list не имеет JSON-режима, поэтому проверяем имя в текстовом выводе.
queues = run("npx", "wrangler", "queues", "list", capture=True)
if "threadsbot-updates" not in queues.stdout:
    run("npx", "wrangler", "queues", "create", "threadsbot-updates")

run("npm", "run", "db:remote")
print("✅ D1, миграции и Queue готовы")


## 4. Безопасная загрузка cookies

Выберите **только JSON-файлы Cookie-Editor/Playwright**. Они попадут в `/content/private_threads_accounts` с правами `0700`, будут импортированы в D1 и сразу удалены. Имена файлов станут именами технических аккаунтов.

Не загружайте cookies в GitHub и не показывайте содержимое файлов в выводе ячеек.


In [ ]:
from google.colab import files
import json, os, shutil
from pathlib import Path

PRIVATE = Path("/content/private_threads_accounts")
if PRIVATE.exists():
    shutil.rmtree(PRIVATE)
PRIVATE.mkdir(mode=0o700)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Файлы не выбраны")
for filename, content in uploaded.items():
    safe_name = Path(filename).name
    if not safe_name.endswith(".json"):
        raise ValueError(f"Разрешены только .json: {safe_name}")
    parsed = json.loads(content)
    if not isinstance(parsed, list) or not parsed:
        raise ValueError(f"Ожидался непустой список cookies: {safe_name}")
    target = PRIVATE / safe_name
    target.write_bytes(content)
    target.chmod(0o600)

try:
    run("npm", "run", "accounts:import", "--", str(PRIVATE), "--remote")
finally:
    # Удаляем cookies даже при ошибке импорта.
    shutil.rmtree(PRIVATE, ignore_errors=True)
    uploaded.clear()
print("✅ Cookies импортированы в D1; временные файлы удалены")


## 5. Необязательный перенос старой SQLite-базы

Запускайте только если хотите сохранить пользователей, подписки, тикеты и аналитику из `bot.db`. Если перенос не нужен — пропустите ячейку.


In [ ]:
from google.colab import files
import os

legacy = files.upload()
if legacy:
    name, content = next(iter(legacy.items()))
    if Path(name).name != "bot.db":
        raise ValueError("Выберите файл bot.db")
    temp_db = Path("/content/private_bot.db")
    temp_db.write_bytes(content); temp_db.chmod(0o600)
    try:
        run("python3", "scripts/migrate_legacy.py", "--db", str(temp_db), "--accounts", "/content/no-accounts")
    finally:
        temp_db.unlink(missing_ok=True)
        legacy.clear()
    print("✅ Старая база перенесена и удалена из Colab")


## 6. Секреты Worker

Значения вводятся скрыто и передаются `wrangler secret put` через stdin. В файлы они не записываются.


In [ ]:
import getpass, secrets, string

def put_secret(name, value):
    subprocess.run(["npx", "wrangler", "secret", "put", name], cwd=PROJECT,
                   input=value + "\n", text=True, check=True,
                   stdout=subprocess.DEVNULL)

telegram_token = getpass.getpass("TELEGRAM_TOKEN: ").strip()
crypto_token = getpass.getpass("CRYPTO_BOT_TOKEN: ").strip()
webhook_secret = getpass.getpass("WEBHOOK_SECRET (Enter = создать автоматически): ").strip()
if not webhook_secret:
    webhook_secret = secrets.token_urlsafe(32)
assert telegram_token and crypto_token

put_secret("TELEGRAM_TOKEN", telegram_token)
put_secret("CRYPTO_BOT_TOKEN", crypto_token)
put_secret("WEBHOOK_SECRET", webhook_secret)
print("✅ Секреты сохранены в Cloudflare")


## 7. Deploy и установка Telegram webhook


In [ ]:
deployed = run("npx", "wrangler", "deploy", capture=True)
output = deployed.stdout + deployed.stderr
print(output)
match = re.search(r'https://[^\s]+\.(?:workers\.dev|pages\.dev)', output)
if not match:
    worker_url = input("Вставьте URL Worker из вывода выше: ").strip().rstrip("/")
else:
    worker_url = match.group(0).rstrip("/")

import requests
response = requests.post(worker_url + "/setup-webhook",
                         headers={"Authorization": "Bearer " + webhook_secret}, timeout=60)
response.raise_for_status()
print("Webhook:", response.json())
health = requests.get(worker_url + "/health", timeout=30)
health.raise_for_status()
print("Health:", health.json())
print("✅ Бот развёрнут")


## 8. Обязательная очистка секретов и runtime

Ячейка удаляет переменные окружения, временные файлы и значения Python. После неё выберите **Runtime → Disconnect and delete runtime**.


In [ ]:
import gc, os, shutil
for key in ("CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID"):
    os.environ.pop(key, None)
for variable in ("telegram_token", "crypto_token", "webhook_secret", "uploaded", "legacy"):
    globals().pop(variable, None)
shutil.rmtree("/content/private_threads_accounts", ignore_errors=True)
Path("/content/private_bot.db").unlink(missing_ok=True)
gc.collect()
print("✅ Локальные секреты очищены. Теперь удалите Colab runtime.")
